In [3]:
import pandas as pd 

In [4]:
venda_caminho = 'db\\Captura_venda_marco.json' 
produto_caminho = 'db\\Configuração Produto.xlsx' 
cliente_caminho = 'db\\Configuração Cliente.xlsx' 

In [5]:
vendas_dados = pd.read_json(venda_caminho, orient='records')
produtos_dados = pd.read_excel(produto_caminho, sheet_name='Produto')
clientes_dados = pd.read_excel(cliente_caminho, sheet_name='Cliente')

In [6]:
vendas_cliente = pd.merge(
    vendas_dados[["DESCRICAOTIPOFATURAMENTO", "VOLUME", "VALORLIQUIDO", "CODIGOCLIENTE", "CODIGOPRODUTO"]],
    clientes_dados[["CODIGOCLIENTE", "DESCRICAOCLIENTE"]],
    how="inner", # Semelhante ao SQL Merge
    on="CODIGOCLIENTE"
    #left_on="CODIGOCLIENTE", --> Para campos com nomes diferentes entre as tabelas
    #right_on="CODIGOCLIENTE" """
)

vendas_completa = pd.merge(
    vendas_cliente[["DESCRICAOCLIENTE", "DESCRICAOTIPOFATURAMENTO", "VOLUME", "VALORLIQUIDO", "CODIGOCLIENTE", "CODIGOPRODUTO"]],
    produtos_dados[["CODIGOPRODUTO", "NOMEPRODUTO", "DESCRICAOGRUPOPRODUTO"]],
    how="inner",
    on="CODIGOPRODUTO"
).drop(columns=["CODIGOPRODUTO", "CODIGOCLIENTE"])

In [7]:
vendas_completa.sample(8)

,DESCRICAOCLIENTE,DESCRICAOTIPOFATURAMENTO,VOLUME,VALORLIQUIDO,NOMEPRODUTO,DESCRICAOGRUPOPRODUTO
1249,DISTRIBUIDORA SABOR DAS BEBIDAS,VENDA,1139,18542.92,Pack Econômico Drink Misto 269ml,Bebidas Mistas Alcoólicas
7710,LOGÍSTICA BEBIDAS DO BRASIL,VENDA,1692,33620.04,Pack Econômico Drink Misto 269ml,Bebidas Mistas Alcoólicas
7690,LOGÍSTICA CENTRAL DE BEBIDAS,VENDA,96,1224.00,Suco de Laranja Integral 500ml,Sucos Naturais Premium
381,CENTRAL DE NUTRIÇÃO E BEBIDAS,VENDA,144,1121.76,Shot Energético Ultra-Concentrado 60ml,Energéticos de Alta Performance
5539,COMERCIAL ALPHA BEBIDAS,VENDA,687,11837.01,Cerveja Artesanal Alpha Gold 600ml,Cervejas Artesanais Premium
3554,CENTRAL CONFORT BEBIDAS,VENDA,1196,9400.56,Soda Limonada Diet 350ml,Bebidas Gasosas Saborizadas
10061,CENTRAL DE NUTRIÇÃO E BEBIDAS,BONIFICAÇÃO,20,68.00,Drink Misto Melancia 269ml,Bebidas Mistas Alcoólicas
11988,DISTRIBUIDORA BRINDE & AMIGOS,VENDA,1447,24092.55,Energético Red Blue Summer Edition 250ml,Bebidas Energéticas Premium


#### Verificando duplicidade de dados

In [8]:
vendas_completa.shape

(12069, 6)

In [9]:
vendas_dados.shape

(12069, 7)

#### Há clientes que não estão na base de vendas?

In [10]:
clientes_faltantes = pd.merge(
    vendas_dados,
    clientes_dados[["CODIGOCLIENTE", "DESCRICAOCLIENTE"]],
    how="right",
    on="CODIGOCLIENTE"
)

In [11]:
clientes_faltantes.isnull().sum()

DATAEMISSAO                 26
CODIGOCLIENTE                0
CODIGOPRODUTO               26
TIPOFATURAMENTO             26
DESCRICAOTIPOFATURAMENTO    26
VOLUME                      26
VALORLIQUIDO                26
DESCRICAOCLIENTE             0
dtype: int64

In [12]:
comparacao = pd.merge(
    vendas_dados,
    clientes_dados[["CODIGOCLIENTE", "DESCRICAOCLIENTE"]],
    how="outer",
    indicator=True
)

comparacao.head(5)

,DATAEMISSAO,CODIGOCLIENTE,CODIGOPRODUTO,TIPOFATURAMENTO,DESCRICAOTIPOFATURAMENTO,VOLUME,VALORLIQUIDO,DESCRICAOCLIENTE,_merge
0,NaN,10000449,NaN,NaN,NaN,NaN,NaN,ATACADO BRINDE & CIA,right_only
1,NaN,10000964,NaN,NaN,NaN,NaN,NaN,LOGÍSTICA ESSENCIAL BEBIDAS,right_only
2,NaN,10001683,NaN,NaN,NaN,NaN,NaN,ATACADISTA BRINDE AMIGO,right_only
3,NaN,10004001,NaN,NaN,NaN,NaN,NaN,LOGÍSTICA GARRAFAS & LOG,right_only
4,2026-03-01T00:00:00.000,10004002,10022282.0,1.0,VENDA,845.0,16773.25,COMERCIAL MUNDO DAS BEBIDAS,both


#### Clientes que não participaram de nossas vendas nesse mês

In [13]:
comparacao = comparacao[comparacao["_merge"] != "both"]

display(comparacao) 

,DATAEMISSAO,CODIGOCLIENTE,CODIGOPRODUTO,TIPOFATURAMENTO,DESCRICAOTIPOFATURAMENTO,VOLUME,VALORLIQUIDO,DESCRICAOCLIENTE,_merge
0,NaN,10000449,NaN,NaN,NaN,NaN,NaN,ATACADO BRINDE & CIA,right_only
1,NaN,10000964,NaN,NaN,NaN,NaN,NaN,LOGÍSTICA ESSENCIAL BEBIDAS,right_only
2,NaN,10001683,NaN,NaN,NaN,NaN,NaN,ATACADISTA BRINDE AMIGO,right_only
3,NaN,10004001,NaN,NaN,NaN,NaN,NaN,LOGÍSTICA GARRAFAS & LOG,right_only
759,NaN,10004003,NaN,NaN,NaN,NaN,NaN,REDE BRINDE FELIZ DISTRIBUIÇÃO,right_only
2992,NaN,10004006,NaN,NaN,NaN,NaN,NaN,SUPRIMENTOS GARRAFAS & COPO,right_only
3826,NaN,10004009,NaN,NaN,NaN,NaN,NaN,ATACADISTA BRINDE SUPREMO,right_only
3827,NaN,10004010,NaN,NaN,NaN,NaN,NaN,ATACADO COPO & GARRAFA,right_only
3828,NaN,10004011,NaN,NaN,NaN,NaN,NaN,DISTRIBUIDORA BEM-ESTAR BEBIDAS,right_only
5452,NaN,10004203,NaN,NaN,NaN,NaN,NaN,ATACADISTA BRINDE FINO,right_only


In [ ]:
comparacao.isna().sum()

DATAEMISSAO                 26
CODIGOCLIENTE               26
CODIGOPRODUTO               26
TIPOFATURAMENTO             26
DESCRICAOTIPOFATURAMENTO    26
VOLUME                      26
VALORLIQUIDO                26
DESCRICAOCLIENTE            26
_merge                      26
dtype: int64